# Self-Organizing Maps (SOMs) Notebook - Alaska Cyclones
### Data extraction

**Notebook by Alice DuVivier - April 2026**
- Pulls selected variable from MESACLIP data that has already been subset to Alaska region.
- Decades will be each decade 1950-2090 and +/-1 year around that date. 15 decades total, or 45 years.
- Want to Pull just SOND data for the decades of interest. 122 days
- 9 ensembles total. Use ens 002-010 *except* for 1951 007 is missing data, so pull ens001 for that year

In [1]:
from glob import glob
import xarray as xr
from datetime import datetime
import time
import cftime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
from datetime import timedelta

# Regridding
import xesmf as xe

# SOM training
#from minisom import MiniSom, asumptotic_decay
#import pickle
#from itertools import product
#import sammon

## Section 1: Load and get correct training data

In [2]:
# set variable to load
var_in = 'PSL'

### Load in the Historical and Future data

In [3]:
# set the input data directory
data_dir = '/glade/campaign/cgd/ppc/duvivier/arctic_actionable/DATA/mesaclip_all_arctic/daily/'+var_in+'/'

In [4]:
%%time
print('loading historical data')

# list 9 members of MESACLIP
# Because over the historical, ensemble 001 doesn't have daily PSL, we load only ens 2-10
ens_mems_hist = ['002','003','004','005','006','007','008','009','010']

ds_hist = xr.Dataset()

for m in ens_mems_hist:
    print(m)

    case = 'MESACLIP_HIST_' + m

    files = sorted(glob(f'{data_dir}/{case}_*_alaska_daily.nc'))        
    ds_tmp = xr.open_mfdataset(files, data_vars="minimal", coords='minimal', compat="override", parallel=True, 
                               concat_dim="time", combine='nested', decode_times=True)
    
    ds_hist = xr.concat([ds_hist,ds_tmp], dim='member_id')

# keep just member_id 1-11 because for whatever reason 0 is missing data
ds_hist = ds_hist.sel(member_id=slice(1,11))

loading historical data
002
003
004
005
006
007
008
009
010
CPU times: user 13.8 s, sys: 2.29 s, total: 16.1 s
Wall time: 54.4 s


In [5]:
%%time
print('loading future data - RCP6.0')

# list 9 members of MESACLIP
# Because over the historical, ensemble 007 is missing year 2051, we replace this ensemble with 001
ens_mems_rcp60 = ['002','003','004','005','006','001','008','009','010']

ds_rcp60 = xr.Dataset()

for m in ens_mems_rcp60:
    print(m)

    case = 'MESACLIP_RCP6.0_' + m

    files = sorted(glob(f'{data_dir}/{case}_*_alaska_daily.nc'))        
    ds_tmp = xr.open_mfdataset(files, data_vars="minimal", coords='minimal', compat="override", parallel=True, 
                               concat_dim="time", combine='nested', decode_times=True)
    
    ds_rcp60 = xr.concat([ds_rcp60,ds_tmp], dim='member_id')

# keep just member_id 1-11 because for whatever reason 0 is missing data
ds_rcp60 = ds_rcp60.sel(member_id=slice(1,11))

loading future data - RCP6.0
002
003
004


<timed exec>:18: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.


005
006


<timed exec>:18: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'cosp_scol' ('cosp_scol',) The recommendation is to set join explicitly for this case.


001
008


<timed exec>:18: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'cosp_scol' ('cosp_scol',) The recommendation is to set join explicitly for this case.


009


<timed exec>:18: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'cosp_scol' ('cosp_scol',) The recommendation is to set join explicitly for this case.


010
CPU times: user 19.8 s, sys: 2.25 s, total: 22 s
Wall time: 1min 15s


<timed exec>:18: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'cosp_scol' ('cosp_scol',) The recommendation is to set join explicitly for this case.


In [6]:
# keep just the variable of interest
ds_hist = ds_hist[var_in]
ds_rcp60 = ds_rcp60[var_in]

In [7]:
# concatenate historical and rcp60
data_all = xr.concat([ds_hist,ds_rcp60], dim='time')

In [8]:
# get number of ensembles
nens = len(data_all.member_id)

### Subset the Decades and Months of interest

In [9]:
# set the decadal array
# 1950 - 2090
decades_all = np.arange(1950,2100,10)
ndec = len(decades_all)

In [10]:
# Keep only the months SOND
data_seasonal = data_all.isel(time=data_all.time.dt.month.isin([9,10,11,12]))

In [11]:
# get some lengths of dimensions

# days are all days in SOND
ndays_seasonal = 30 + 31 + 30 + 31

# number of latitude points
nlat = len(data_seasonal.lat)
#print(nlat)
# number of longitude points
nlon = len(data_seasonal.lon)
#print(nlon)

In [12]:
# preallocate arrays
array_tmp = np.zeros([nens,ndec,3*ndays_seasonal,nlat,nlon])

# loop through ensembles
for d in np.arange(0,ndec,1):
    print('getting data for: '+str(decades_all[d]))
    # get array of years to load
    yrs_load = np.arange(decades_all[d]-1,decades_all[d]+2,1)
    #print(yrs_load)
    # Keep only these years
    data_sub = data_seasonal.isel(time=data_seasonal.time.dt.year.isin(yrs_load))
    array_tmp[:,d,:,:,:] = data_sub

# convert the numpy array to xarray for easier manipulation
data_subset = xr.DataArray(array_tmp,dims=('ensemble','decades','autumn_days','lat','lon'))
data_subset['decades'] = decades_all
data_subset['lat'] = data_seasonal.lat
data_subset['lon'] = data_seasonal.lon


getting data for: 1950
getting data for: 1960
getting data for: 1970
getting data for: 1980
getting data for: 1990
getting data for: 2000
getting data for: 2010
getting data for: 2020
getting data for: 2030
getting data for: 2040
getting data for: 2050
getting data for: 2060
getting data for: 2070
getting data for: 2080
getting data for: 2090


In [13]:
data_subset

<xarray.DataArray (ensemble: 9, decades: 15, autumn_days: 366, lat: 160,
                   lon: 240)> Size: 15GB
array([[[[[101766.77838429, 101774.59887581, 101788.95449184, ...,
           101591.8384898 , 101579.99954082, 101570.56771114],
          [101733.67370182, 101740.11971448, 101747.4879611 , ...,
           101585.05759591, 101577.61712429, 101561.26528599],
          [101694.63914163, 101710.76756553, 101720.10231565, ...,
           101578.86879103, 101568.25908934, 101563.19407114],
          ...,
          [102305.38498289, 102305.47240942, 102305.5574262 , ...,
           102291.76604624, 102291.12700657, 102290.47564175],
          [102270.1298087 , 102270.2373595 , 102270.34852466, ...,
           102240.37184641, 102240.45265849, 102240.53216708],
          [102234.62566511, 102234.62898868, 102234.63222065, ...,
           102234.67166155, 102234.62183037, 102234.57094863]],

         [[101304.79524836, 101307.40773527, 101303.08003849, ...,
           101274.93723838, 101256.92470309, 101246.6840838 ],
          [101237.28320823, 101231.30979275, 101225.03446644, ...,
           101258.51140025, 101249.85854898, 101230.04470959],
          [101172.98618758, 101170.97439524, 101174.70640933, ...,
           101245.30291628, 101229.71176061, 101223.53621236],
...
           100029.18826367, 100029.79868962, 100030.42088885],
          [ 99931.51319103,  99932.01282834,  99932.48384029, ...,
           100009.16175169, 100009.77441302, 100010.37719212],
          [ 99977.6580736 ,  99977.65412587,  99977.65028695, ...,
           100009.56493813, 100009.64379798, 100009.72337213]],

         [[ 99234.53122991,  99251.91995266,  99296.86193732, ...,
           101384.57198316, 101392.65395754, 101405.34584171],
          [ 99196.21169157,  99216.80699956,  99240.280775  , ...,
           101355.07430738, 101370.66058861, 101392.59203286],
          [ 99148.22944806,  99190.39996225,  99211.29703069, ...,
           101339.16729439, 101354.30839393, 101362.39536252],
          ...,
          [100869.96370881, 100869.92629763, 100869.88991761, ...,
           100820.24775303, 100820.53559339, 100820.82898532],
          [100876.70125776, 100876.13788875, 100875.50522171, ...,
           100864.14665157, 100863.90315796, 100863.66359191],
          [100878.92581585, 100878.92474467, 100878.92370303, ...,
           100868.03833933, 100868.04588627, 100868.05393995]]]]],
      shape=(9, 15, 366, 160, 240))
Coordinates:
  * decades  (decades) int64 120B 1950 1960 1970 1980 ... 2060 2070 2080 2090
  * lat      (lat) float64 1kB 50.12 50.38 50.62 50.88 ... 89.38 89.62 89.88
  * lon      (lon) float64 2kB 170.1 170.4 170.6 170.9 ... 229.4 229.6 229.9
Dimensions without coordinates: ensemble, autumn_days

## Save data as a netcdf

In [14]:
# set the output data directory
data_dir = '/glade/campaign/cgd/ppc/duvivier/arctic_actionable/DATA/SOM_alaska_cyclones/data_for_composites/'

In [19]:
fout = data_dir + 'PSL_autumn_decadal_alaska_cam_grid.nc'

In [24]:
# make output array
ds_out = data_subset.to_dataset(name='psl')

# change the attributes
ds_out.attrs['author'] = 'Alice DuVivier'
ds_out.attrs['date_processed'] = datetime.now().strftime('%Y-%m-%d')
ds_out.attrs['contents'] = f'PSL Autumn daily MESACLIP Data by decade'

In [25]:
ds_out

<xarray.Dataset> Size: 15GB
Dimensions:  (decades: 15, lat: 160, lon: 240, ensemble: 9, autumn_days: 366)
Coordinates:
  * decades  (decades) int64 120B 1950 1960 1970 1980 ... 2060 2070 2080 2090
  * lat      (lat) float64 1kB 50.12 50.38 50.62 50.88 ... 89.38 89.62 89.88
  * lon      (lon) float64 2kB 170.1 170.4 170.6 170.9 ... 229.4 229.6 229.9
Dimensions without coordinates: ensemble, autumn_days
Data variables:
    psl      (ensemble, decades, autumn_days, lat, lon) float64 15GB 1.018e+0...
Attributes:
    author:          Alice DuVivier
    date_processed:  2026-04-08
    contents:        PSL Autumn daily MESACLIP Data by decade

In [26]:
# calculate the size of the dataset in GB
size_gb = ds_out.nbytes / (1024**3)
print(f"The dataset is approximately {size_gb:.2f} GB.")

The dataset is approximately 14.14 GB.


In [27]:
# Export the dataset to NetCDF with all attributes and coordinates
print('Writing post processed file')
ds_out.to_netcdf(fout)

Writing post processed file
